# IMPORT

In [1]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# DEFINE FOLDERS

In [2]:
STATION_CLUSTER_FOLDER = Path("station_clusters")
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
MODEL_OUTPUT_FOLDER = Path("cluster_models")

folders = [STATION_CLUSTER_FOLDER, DATASET_INPUT_FOLDER, MODEL_OUTPUT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA AND CLUSTERS

In [ ]:

df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/station_clusters_2_42.csv")

# Merge cluster info
df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)


# FEATURE SELECTION

In [4]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Weather
    "weather_temperature_2m",
    "weather_wind_speed_10m",
    "weather_weather_code",
    "weather_is_day",

    # Station context
    "capacity",
]

TARGET = "total_bikes_now"


# TIME BASED 70-15-15 SPLIT

In [5]:
def time_split(df, train_frac=0.7, val_frac=0.15):
    df = df.sort_values("duedate")

    n = len(df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train = df.iloc[:train_end]
    val   = df.iloc[train_end:val_end]
    test  = df.iloc[val_end:]

    return train, val, test


# TRAIN MODELS PER CLUSTER

In [6]:
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"\n🔹 Training cluster {cluster_id}")

    df_c = df[df["cluster"] == cluster_id]

    X = df_c[FEATURES]
    y = df_c[TARGET]

    # Train / test split (time-agnostic for now)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} → R²={r2:.3f} | MAE={mae:.2f}")



🔹 Training cluster 0
  LinearRegression → R²=0.338 | MAE=7.75
  SGDRegressor     → R²=0.338 | MAE=7.75
  RandomForest     → R²=0.425 | MAE=7.22

🔹 Training cluster 1
  LinearRegression → R²=0.171 | MAE=4.78
  SGDRegressor     → R²=0.170 | MAE=4.78
  RandomForest     → R²=0.275 | MAE=4.48


# COMPARE RESULTS

In [7]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,n_samples
2,0,RandomForest,0.425368,7.223860,2309781
0,0,LinearRegression,0.338124,7.752850,2309781
1,0,SGDRegressor,0.337982,7.752980,2309781
5,1,RandomForest,0.275329,4.480552,2453226
3,1,LinearRegression,0.171115,4.781576,2453226
4,1,SGDRegressor,0.170498,4.784709,2453226


# GET BEST MODEL PER CLUSTER

In [8]:
best_models = {}

for cluster_id in results_df["cluster"].unique():
    best = results_df[results_df["cluster"] == cluster_id].sort_values(
        "r2", ascending=False
    ).iloc[0]

    best_models[cluster_id] = best["model"]

print(best_models)

{np.int64(0): 'RandomForest', np.int64(1): 'RandomForest'}


# SAVE BEST MODEL PER CLUSTER

In [9]:
for cluster_id in best_models:
    df_c = df[df["cluster"] == cluster_id]
    X = df_c[FEATURES]
    y = df_c[TARGET]

    model_name = best_models[cluster_id]

    if model_name == "LinearRegression":
        model = LinearRegression()
    elif model_name == "SGDRegressor":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ])
    else:
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        )

    model.fit(X, y)
    joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/model_cluster_{cluster_id}.pkl")

print("Models saved per cluster")


✅ Models saved per cluster
